# Explicabilité réglementaire — SHAP par sous-groupe (M2 et M4)

**Ce que ce notebook ajoute.** `extraire_coefficients` (utilisée depuis `baseline.ipynb`) donne un
classement *global* des variables par poids de coefficient — le même pour tous les clients. Ici, on
calcule des **valeurs SHAP par client** (`shap.LinearExplainer`, exact pour un modèle linéaire comme
notre `LogisticRegression`) puis on les agrège **par sous-groupe** (`secteur` formel/informel, `sexe`)
pour vérifier si une variable pèse très différemment d'un profil à l'autre — un niveau de détail que le
classement global ne montre pas, et qui compte pour l'explicabilité réglementaire (justifier une
décision *pour un client donné*, pas seulement en moyenne sur le portefeuille) et pour le repérage d'un
biais indirect (une variable qui ne discrimine pas explicitement par secteur/sexe mais dont le poids
diffère fortement entre groupes).

**Portée : M2 et M4 seulement.** M6 (agent LLM, `m5_m6_llm.ipynb`) est délibérément exclu de ce
notebook : les scores S1/S2/S3 du pilote à 60 dossiers n'ont pas été persistés sur disque (seuls des
résumés statistiques le sont, dans les sorties déjà exécutées de `m5_m6_llm.ipynb`), et les reproduire
demanderait de relancer l'agent LLM local (~45-60 min au débit déjà mesuré, ~31-33 s/appel) — un coût
jugé disproportionné pour l'instant face au gain (décision validée avec l'utilisateur le 2026-08-23).
Le SHAP de M6 est une suite naturelle une fois les scores LLM persistés dans un run ultérieur de
`m5_m6_llm.ipynb`.

**Écarts au protocole déjà assumés dans `m2_m3_texte.ipynb` et `m4_embeddings.ipynb`** (pas de
mensonge silencieux, hérités tels quels ici : `camembert-base` à la place de
`paraphrase-multilingual-MiniLM-L12-v2`, faute d'accès internet au moment de l'exécution) s'appliquent
identiquement à la reconstruction de M4 ci-dessous.

**Choix méthodologique — `LinearExplainer` plutôt que `Kernel`/`Permutation`.** Le classifieur final de
chaque palier est un `LogisticRegression` sur les variables déjà transformées par le
`ColumnTransformer` du pipeline (mise à l'échelle, one-hot, SVD du texte le cas échéant) — un modèle
linéaire dans cet espace transformé. `shap.LinearExplainer` calcule alors les valeurs SHAP de façon
**exacte** (pas d'échantillonnage/approximation comme pour un modèle boîte noire), en travaillant sur
la sortie *log-odds* du modèle (le même espace que les coefficients d'`extraire_coefficients`) par
rapport à une distribution de référence (`X_fond`, un sous-échantillon du train — voir
`calculer_shap` dans `scoring_utils.py`).

In [1]:
import re

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from transformers import CamembertModel, CamembertTokenizer, logging as hf_logging

from categorisation import construire_parts_categories
from scoring_utils import (
    calculer_shap,
    construire_pipeline,
    construire_variables_comportementales,
    evaluer,
    extraire_coefficients,
    rechercher_meilleur_C,
    shap_par_sousgroupe,
)

hf_logging.set_verbosity_error()  # les poids du pooler non utilisés déclenchent un avertissement sans objet ici

SEED = 42
TARGET = "defaut_90j"

DECLARATIF_NUM = ["age", "revenu_declare", "anciennete_mois"]
DECLARATIF_CAT = ["zone", "secteur", "region"]
COMPORTEMENTAL_NUM = [
    "nb_tx", "pct_debits", "inflow", "outflow", "net_flow",
    "mean_abs", "std_abs", "max_abs", "cv_abs",
    "nb_jours_actifs", "tx_par_jour", "ecart_revenu",
]
GRILLE_C = [0.001, 0.01, 0.1, 1, 10, 100]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device (embeddings CamemBERT) : {DEVICE}")

device (embeddings CamemBERT) : cuda


## Étape 1 — Charger les données

Mêmes fichiers que les notebooks précédents.

In [2]:
clients = pd.read_csv("clients_synth.csv")
tx = pd.read_csv("transactions_synth.csv")

print(f"clients : {clients.shape[0]} lignes | transactions : {tx.shape[0]} lignes")

clients : 2000 lignes | transactions : 91666 lignes


## Étape 2 — Variables comportementales (M1, rappel)

`construire_variables_comportementales` (détaillée dans `baseline.ipynb`), sans lire le sens des
libellés — base commune à M2 et M4.

In [3]:
comp = construire_variables_comportementales(tx, clients)
comp.head()

,client_id,nb_tx,pct_debits,inflow,outflow,mean_abs,std_abs,max_abs,nb_jours_actifs,net_flow,cv_abs,tx_par_jour,ecart_revenu
0,1,28,0.821429,1970486.0,1958456.0,140319.357143,352120.284243,1481199.0,26,12030.0,2.509421,1.076923,-5.025951
1,2,58,0.810345,4128443.0,2782051.0,119146.448276,251905.005755,1009317.0,44,1346392.0,2.114247,1.318182,-6.908895
2,3,36,0.750000,2775876.0,2040831.0,133797.416667,292261.844375,1364223.0,25,735045.0,2.184361,1.440000,-3.895725
3,4,21,0.857143,179598.0,1871038.0,97649.333333,319716.352365,1489911.0,19,-1691440.0,3.274127,1.105263,-0.814121
4,5,29,0.931034,397596.0,4884317.0,182134.931034,357177.292957,1209253.0,20,-4486721.0,1.961059,1.450000,0.181901


## Étape 3 — Reconstruire et entraîner M2 (règles lexicales)

Reprise résumée de `m2_m3_texte.ipynb` : même construction, même split (`SEED=42, test_size=0.20`),
mêmes clients en test que dans tous les autres notebooks.

In [4]:
parts_m2, tx_categorise_m2 = construire_parts_categories(tx)
PART_COLS_M2 = [c for c in parts_m2.columns if c.startswith("PART_")]

precision_m2 = (tx_categorise_m2.cat_regle == tx_categorise_m2.gt_categorie).mean()
print(f"précision de la catégorisation par règles (M2) : {precision_m2:.3f}")

précision de la catégorisation par règles (M2) : 0.710


In [5]:
df_m2 = (clients
         .merge(comp, on="client_id", how="left")
         .merge(parts_m2, on="client_id", how="left"))
df_m2[COMPORTEMENTAL_NUM + PART_COLS_M2] = df_m2[COMPORTEMENTAL_NUM + PART_COLS_M2].fillna(0)

y = df_m2[TARGET].values
colonnes_gt = [c for c in df_m2.columns if c.startswith("gt_")]
X_m2 = df_m2.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr_m2, Xte_m2, ytr_m2, yte_m2 = train_test_split(
    X_m2, y, test_size=0.20, stratify=y, random_state=SEED
)
dfte_m2 = df_m2.loc[Xte_m2.index]

print(f"train {len(Xtr_m2)} | test {len(Xte_m2)}")

train 1600 | test 400


In [6]:
num_m2 = DECLARATIF_NUM + COMPORTEMENTAL_NUM + PART_COLS_M2

pipeline_m2 = construire_pipeline(num_m2, DECLARATIF_CAT, seed=SEED)
modele_m2, _ = rechercher_meilleur_C(pipeline_m2, Xtr_m2, ytr_m2, GRILLE_C, seed=SEED)

p2 = modele_m2.predict_proba(Xte_m2)[:, 1]
_ = evaluer("M2", yte_m2, p2)

extraire_coefficients(modele_m2, num_m2, DECLARATIF_CAT).head(15)

M2   | AUC 0.718 | Gini 0.436 | KS 0.351


,variable,coefficient
0,secteur_formel,-0.260984
1,secteur_informel,0.260892
2,PART_AGIOS,0.192302
3,tx_par_jour,-0.162729
4,PART_NJANGUI,-0.154008
5,PART_LOYER,0.147819
6,revenu_declare,-0.123059
7,nb_jours_actifs,-0.114624
8,outflow,-0.110242
9,cv_abs,0.098078


## Étape 4 — SHAP du modèle M2

`calculer_shap` transforme `Xtr_m2`/`Xte_m2` par le préprocesseur déjà entraîné, calcule les valeurs
SHAP du `LogisticRegression` (espace log-odds) pour chaque client du test, avec le train comme
distribution de référence. Le classement global (moyenne de `|SHAP|` sur le test) doit être cohérent
avec le classement par coefficient de l'étape précédente — c'est la même information, mais ici
disponible **par client**, ce qui permet l'agrégation par sous-groupe plus bas.

In [7]:
valeurs_shap_m2, base_m2 = calculer_shap(modele_m2, Xtr_m2, Xte_m2, seed=SEED)

print(f"valeur de base (log-odds moyen du fond de référence) : {base_m2:.3f}\n")
valeurs_shap_m2.abs().mean().sort_values(ascending=False).head(15).rename("moyenne |SHAP|").to_frame()

valeur de base (log-odds moyen du fond de référence) : -0.313



,moyenne |SHAP|
PART_AGIOS,0.148634
tx_par_jour,0.134109
secteur_formel,0.130048
secteur_informel,0.130003
PART_LOYER,0.117844
PART_NJANGUI,0.114079
revenu_declare,0.098734
nb_jours_actifs,0.096962
outflow,0.094429
cv_abs,0.076378


**Comparaison par sous-groupe.** `shap_par_sousgroupe` limite la comparaison aux variables les
plus importantes au global (évite de comparer du bruit), et donne la moyenne de `|SHAP|` dans chaque
sous-groupe pour chacune — une variable dont les colonnes divergent fortement d'un groupe à l'autre
pèse différemment dans la décision selon le profil du client, même si elle n'est pas elle-même une
variable de secteur ou de sexe.

In [8]:
shap_par_sousgroupe(valeurs_shap_m2, dfte_m2, "secteur")

_groupe,formel,informel
PART_AGIOS,0.132127,0.162555
tx_par_jour,0.135941,0.132565
secteur_formel,0.135712,0.125272
secteur_informel,0.135664,0.125228
PART_LOYER,0.127048,0.110081
PART_NJANGUI,0.131927,0.099028
revenu_declare,0.095058,0.101834
nb_jours_actifs,0.101039,0.093524
outflow,0.104526,0.085913
cv_abs,0.074835,0.077679


In [9]:
shap_par_sousgroupe(valeurs_shap_m2, dfte_m2, "sexe")

_groupe,F,M
PART_AGIOS,0.140813,0.157366
tx_par_jour,0.128000,0.140929
secteur_formel,0.130220,0.129857
secteur_informel,0.130174,0.129811
PART_LOYER,0.113819,0.122337
PART_NJANGUI,0.118061,0.109635
revenu_declare,0.096471,0.101260
nb_jours_actifs,0.093434,0.100901
outflow,0.088934,0.100563
cv_abs,0.077383,0.075256


## Étape 5 — Recalculer embeddings + clustering (M4, rappel)

Reprise résumée de `m4_embeddings.ipynb`, à l'échelle complète (pas le sous-échantillon pilote de
`m5_m6_llm.ipynb`) : nettoyage léger pour préserver les frontières de mots, embeddings CamemBERT par
mean-pooling, réduction PCA puis K-Means avec K choisi par silhouette sur la même grille.

In [10]:
def nettoyer_pour_embedding(libelle):
    """Nettoyage léger : garde les mots et leur ordre, retire le bruit technique."""
    s = str(libelle).lower()
    s = re.sub(r"(ref|tpe|ag)\w*", " ", s)
    s = re.sub(r"\d+", " ", s)
    s = re.sub(r"[^a-zàâäéèêëïîôöùûüç\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
modele_embed = CamembertModel.from_pretrained("camembert-base").to(DEVICE).eval()


def embarquer_textes(textes, batch_size=64):
    """Embedding de phrase par mean-pooling des dernières couches cachées de CamemBERT."""
    vecteurs = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        entrees = tokenizer(lot, padding=True, truncation=True, max_length=64,
                             return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            sortie = modele_embed(**entrees).last_hidden_state
        masque = entrees["attention_mask"].unsqueeze(-1).float()
        moyenne = (sortie * masque).sum(1) / masque.sum(1).clamp(min=1e-9)
        vecteurs.append(moyenne.cpu().numpy())
    return np.vstack(vecteurs)


tx["libelle_net"] = tx.libelle.map(nettoyer_pour_embedding)
uniques_df = pd.DataFrame({"libelle_net": tx.libelle_net.drop_duplicates().reset_index(drop=True)})
print(f"{len(uniques_df)} libellés uniques (nettoyés) sur {len(tx)} transactions")

embeddings_uniques = embarquer_textes(uniques_df.libelle_net.tolist())
embeddings_uniques.shape

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

33040 libellés uniques (nettoyés) sur 91666 transactions


(33040, 768)

In [11]:
pca = PCA(n_components=50, random_state=SEED)
embeddings_reduits = pca.fit_transform(embeddings_uniques)
print(f"variance expliquée par les 50 composantes : {pca.explained_variance_ratio_.sum():.1%}")

rng = np.random.default_rng(SEED)
grille_K = [5, 8, 10, 12, 15, 18, 20, 25]
resultats_K = []
for k in grille_K:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(embeddings_reduits)
    if len(embeddings_reduits) > 3000:
        idx = rng.choice(len(embeddings_reduits), 3000, replace=False)
        sil = silhouette_score(embeddings_reduits[idx], km.labels_[idx])
    else:
        sil = silhouette_score(embeddings_reduits, km.labels_)
    resultats_K.append({"K": k, "inertie": km.inertia_, "silhouette": sil})

resultats_K = pd.DataFrame(resultats_K)
resultats_K

variance expliquée par les 50 composantes : 83.6%


,K,inertie,silhouette
0,5,58842.558594,0.102304
1,8,54019.992188,0.075896
2,10,52237.148438,0.066086
3,12,50813.992188,0.070249
4,15,49064.074219,0.071947
5,18,47646.531250,0.064032
6,20,46832.378906,0.064959
7,25,45142.300781,0.061862


In [12]:
K_FINAL = int(resultats_K.loc[resultats_K.silhouette.idxmax(), "K"])
print(f"K retenu (meilleur silhouette) : {K_FINAL}")

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=10).fit(embeddings_reduits)
uniques_df["cluster"] = kmeans_final.labels_

K retenu (meilleur silhouette) : 5


In [13]:
tx_clusterise = tx.merge(uniques_df, on="libelle_net", how="left")

noms_clusters = (tx_clusterise.groupby("cluster").gt_categorie
                  .agg(lambda s: s.value_counts().idxmax()))

tx_clusterise["nom_cluster"] = tx_clusterise.cluster.map(noms_clusters)
precision_m4 = (tx_clusterise.nom_cluster == tx_clusterise.gt_categorie).mean()

print(f"précision de reconstitution des catégories : {precision_m4:.3f} par clustering (M4) "
      f"vs {precision_m2:.3f} par règles (M2)\n")
noms_clusters.rename("nom_majoritaire (gt_categorie)").to_frame()

précision de reconstitution des catégories : 0.179 par clustering (M4) vs 0.710 par règles (M2)



,nom_majoritaire (gt_categorie)
cluster,
0,MOMO
1,SALAIRE
2,SALAIRE
3,TRANSPORT
4,MOMO


In [14]:
parts_clust = (tx_clusterise.groupby(["client_id", "cluster"]).size()
               .unstack(fill_value=0))
parts_clust = parts_clust.div(parts_clust.sum(axis=1), axis=0)
parts_clust.columns = [f"PART_CLUST_{c}" for c in parts_clust.columns]
parts_clust = parts_clust.reset_index()
PART_CLUST_COLS = [c for c in parts_clust.columns if c.startswith("PART_CLUST_")]

parts_clust.head()

,client_id,PART_CLUST_0,PART_CLUST_1,PART_CLUST_2,PART_CLUST_3,PART_CLUST_4
0,1,0.142857,0.071429,0.107143,0.428571,0.250000
1,2,0.258621,0.086207,0.224138,0.189655,0.241379
2,3,0.305556,0.000000,0.222222,0.250000,0.222222
3,4,0.190476,0.047619,0.190476,0.238095,0.333333
4,5,0.310345,0.103448,0.172414,0.172414,0.241379


## Étape 6 — Construire X/y et entraîner M4

Même split que M2 (et que tous les autres notebooks) — les mêmes clients en test partout.

In [15]:
df_m4 = (clients
         .merge(comp, on="client_id", how="left")
         .merge(parts_m2, on="client_id", how="left")
         .merge(parts_clust, on="client_id", how="left"))
colonnes_a_remplir = COMPORTEMENTAL_NUM + PART_COLS_M2 + PART_CLUST_COLS
df_m4[colonnes_a_remplir] = df_m4[colonnes_a_remplir].fillna(0)

y = df_m4[TARGET].values
colonnes_gt = [c for c in df_m4.columns if c.startswith("gt_")]
X_m4 = df_m4.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr_m4, Xte_m4, ytr_m4, yte_m4 = train_test_split(
    X_m4, y, test_size=0.20, stratify=y, random_state=SEED
)
dfte_m4 = df_m4.loc[Xte_m4.index]

print(f"train {len(Xtr_m4)} | test {len(Xte_m4)}")

train 1600 | test 400


In [16]:
num_m4 = DECLARATIF_NUM + COMPORTEMENTAL_NUM + PART_CLUST_COLS

pipeline_m4 = construire_pipeline(num_m4, DECLARATIF_CAT, seed=SEED)
modele_m4, _ = rechercher_meilleur_C(pipeline_m4, Xtr_m4, ytr_m4, GRILLE_C, seed=SEED)

p4 = modele_m4.predict_proba(Xte_m4)[:, 1]
_ = evaluer("M4", yte_m4, p4)

extraire_coefficients(modele_m4, num_m4, DECLARATIF_CAT).head(15)

M4   | AUC 0.681 | Gini 0.361 | KS 0.297


,variable,coefficient
0,secteur_formel,-0.293265
1,secteur_informel,0.293137
2,tx_par_jour,-0.154258
3,nb_jours_actifs,-0.137485
4,cv_abs,0.116283
5,nb_tx,-0.105567
6,outflow,-0.104925
7,inflow,-0.101962
8,ecart_revenu,0.073980
9,anciennete_mois,-0.067607


## Étape 7 — SHAP du modèle M4

Même méthode qu'à l'Étape 4, sur le pipeline M4 (`PART_CLUST_*` à la place des `PART_*` de règles).

In [17]:
valeurs_shap_m4, base_m4 = calculer_shap(modele_m4, Xtr_m4, Xte_m4, seed=SEED)

print(f"valeur de base (log-odds moyen du fond de référence) : {base_m4:.3f}\n")
valeurs_shap_m4.abs().mean().sort_values(ascending=False).head(15).rename("moyenne |SHAP|").to_frame()

valeur de base (log-odds moyen du fond de référence) : -0.276



,moyenne |SHAP|
secteur_formel,0.146134
secteur_informel,0.146070
tx_par_jour,0.127128
nb_jours_actifs,0.116300
cv_abs,0.090555
outflow,0.089874
nb_tx,0.087474
inflow,0.080770
ecart_revenu,0.061815
anciennete_mois,0.058641


In [18]:
shap_par_sousgroupe(valeurs_shap_m4, dfte_m4, "secteur")

_groupe,formel,informel
secteur_formel,0.152498,0.140767
secteur_informel,0.152431,0.140706
tx_par_jour,0.128864,0.125664
nb_jours_actifs,0.121190,0.112177
cv_abs,0.088726,0.092097
outflow,0.099484,0.081769
nb_tx,0.089642,0.085645
inflow,0.091730,0.071527
ecart_revenu,0.064720,0.059365
anciennete_mois,0.062504,0.055383


In [19]:
shap_par_sousgroupe(valeurs_shap_m4, dfte_m4, "sexe")

_groupe,F,M
secteur_formel,0.146327,0.145919
secteur_informel,0.146263,0.145855
tx_par_jour,0.121337,0.133593
nb_jours_actifs,0.112068,0.121025
cv_abs,0.091746,0.089224
outflow,0.084644,0.095713
nb_tx,0.083606,0.091792
inflow,0.078084,0.083767
ecart_revenu,0.061079,0.062637
anciennete_mois,0.057247,0.060197


## Lecture des résultats

**À lire une fois les cellules ci-dessus exécutées** (ce notebook ne préjuge pas du résultat avant
exécution réelle) :

- Le classement global par `|SHAP|` moyen (Étapes 4 et 7) doit rester cohérent avec le classement par
  coefficient (`extraire_coefficients`, déjà vu dans `m2_m3_texte.ipynb` et `m4_embeddings.ipynb`) —
  c'est un contrôle de cohérence, pas un résultat nouveau en soi.
- Le résultat propre à ce notebook est la **comparaison par sous-groupe** (`secteur`, `sexe`) : une
  variable dont la moyenne de `|SHAP|` diffère fortement d'un groupe à l'autre pèse différemment dans
  la décision selon le profil du client — à interpréter avec `equite.ipynb` (biais déjà connu sur
  `secteur`) plutôt qu'isolément.

**Limites assumées :**
- **M6 non couvert** (voir l'introduction) — le SHAP de l'agent LLM reste à faire une fois les scores
  S1/S2/S3 du pilote persistés sur disque dans un run ultérieur de `m5_m6_llm.ipynb`.
- **`LinearExplainer` suppose un modèle linéaire dans l'espace transformé** (vrai ici par construction
  du pipeline, mais signifie que les valeurs SHAP n'ajoutent aucune information sur des interactions
  non linéaires entre variables — pour M4, les `PART_CLUST_*` sont déjà le résultat d'un clustering non
  linéaire en amont, mais le classifieur final reste linéaire sur ces parts).
- **Distribution de référence (`X_fond`)** sous-échantillonnée à 200 lignes du train (`calculer_shap`,
  `echantillon_fond=200`) plutôt que le train entier, pour rester rapide — un choix de vitesse, pas une
  contrainte de `LinearExplainer` (qui accepterait le train complet).
- **Comparaison par sous-groupe non testée statistiquement** ici (pas de test formel du type
  « l'écart de `|SHAP|` moyen entre secteurs est-il significatif ? ») — les tableaux se lisent comme un
  diagnostic exploratoire, pas comme une conclusion causale.